# EMI accuracy distribution: unimodality assessment

Tests whether the per-subject EMI balanced-accuracy distribution departs from unimodality (Hartigan's dip test), with supporting descriptive statistics. Analysis functions are in `analysis/emi_bimodality.py`. The EMI vector is verified against the reported Table I values (86.9 / 9.8) before analysis.

In [ ]:
import sys, os
from pathlib import Path
_here = Path('.').resolve()
repo_root = next((p for p in [_here, _here.parent, _here.parent.parent]
                  if (p / 'config.yaml').exists()), _here)
os.chdir(repo_root); sys.path.insert(0, str(repo_root))

%matplotlib inline
import json
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['figure.dpi'] = 120

from analysis import emi_modality as emb

## Load EMI vector and verify against Table I

In [ ]:
subject_ids, emi, source_files = emb.load_emi_accuracy('data/derived')
print(f'N = {len(emi)}; mean = {emi.mean():.2f}; SD = {emi.std(ddof=1):.2f}; '
      f'min = {emi.min():.2f}; max = {emi.max():.2f}')
print(emb.verify_against_reported(emi))

Optional cross-check against the canonical Table-I loader (requires the classifier stack; skipped where absent).

In [ ]:
try:
    from analysis.classifier import load_all_results, build_accuracy_dataframe
    _df = build_accuracy_dataframe(load_all_results('data/derived'))
    _emi = (_df[_df['condition'] == 'emi']
            .set_index('subject')['balanced_accuracy']
            .reindex(subject_ids).to_numpy())
    assert np.allclose(_emi, emi, atol=1e-9)
    print('Cross-check passed: identical to build_accuracy_dataframe.')
except ModuleNotFoundError as e:
    print(f'Classifier stack unavailable ({e.name}); relying on the Table I guard.')

## Analysis

In [ ]:
results = emb.analyze_emi(emi, subject_ids, source_files=source_files)
print(json.dumps({k: results[k] for k in
                  ['summary', 'dip_test', 'bimodality_coefficient']}, indent=2))

In [ ]:
print(json.dumps(results['collapse_and_normality'], indent=2))

## Figure

In [ ]:
fig, ax = emb.plot_distribution(
    subject_ids, emi, results,
    save_path='data/derived/emi-bimodality/emi_accuracy_distribution.png')
plt.show()

In [ ]:
out = emb.save_results(results, 'data/derived/emi-bimodality/emi_bimodality_results.json')
print(f'saved -> {out}')
print(json.dumps(results['provenance'], indent=2))